# Example: Symmetry-reduced Pulse Design Tool (MAST-U-like)

---

This notebook demonstrates how to use `SymmetryReducedSolver` with the **FreeGSNKE Pulse Design Tool** (FPDT) on a MAST-U-like tokamak, evolving the coupled plasma/metal system strictly in the even-in-$Z$ subspace. It repeats the closed-loop simulation of Example 12b on the MAST-U-like machine with the plasma-control-system (PCS) settings of Example 11, and adds a feedback-controlled sweep of the lower outer divertor strikepoint radius from a conventional position ($R_{\mathrm{sp}} \approx 0.74\text{ m}$) out to a Super-X configuration ($R_{\mathrm{sp}} = 1.40\text{ m}$).

This enables large timesteps to be taken ($5\times$ larger than the $0.5\text{ ms}$ timestep used in Example 11) for rapid prototyping of plasma scenarios.

In this notebook, we:
1. Prepare an exactly up-down symmetric version of the MAST-U-like machine.
2. Construct a symmetric initial Grad-Shafranov equilibrium.
3. Build the virtual circuits for the PCS (including the lower outer strikepoint radius) and configure it with the settings of Example 11.
4. Inspect the even and odd passive structure modes.
5. Run a closed-loop FPDT simulation using `SymmetryReducedSolver` in place of `nl_solver`.

**Note:** the MAST-U-like machine is only approximately up-down symmetric, so symmetrising it changes the machine slightly. The vertical-control circuit `P6` is electrically odd and is removed in the process, so this simulation has no active vertical control: the plasma is held on the midplane by the symmetry itself, and the vertical instability cannot be studied (see Example 11 for that).

### 1. Build the symmetrised MAST-U-like machine

We load the same MAST-U-like description as Example 11 and use `prepare_up_down_symmetric_machine` (see Example 12a) to turn it into an exactly up-down symmetric one. Three things are worth noting:

- The active coils are series-connected upper/lower pairs, so each is already a single even circuit. The exception is `P6`, whose lower bundle has the opposite polarity: it is electrically odd, so we exclude it explicitly (`exclude_odd_active=True`).
- The upper and lower passive structures, limiter, and wall agree only to a few millimetres, with a small common vertical offset. With `z_midplane="auto"` the offset is fitted and removed, and the paired elements are then averaged. Pairs are found automatically from their names and reflected geometry.
- The limiter and wall are intersected with their reflections, which removes a small area (below 0.3%) from each.

In [ ]:
import pickle
import time

import matplotlib.pyplot as plt
import numpy as np

from freegsnke import GSstaticsolver, build_machine, equilibrium_update
from freegsnke.jtor_update import Lao85
from freegsnke.up_down_symmetry import prepare_up_down_symmetric_machine

In [ ]:
# load the (approximately symmetric) MAST-U-like machine description used in Example 11
machine_path = "../machine_configs/MAST-U/MAST-U_like_"
machine_data = {}
for part in ("active_coils", "passive_coils", "limiter", "wall"):
    with open(f"{machine_path}{part}.pickle", "rb") as file:
        machine_data[part] = pickle.load(file)

# exactly up-down symmetric machine data (the odd circuit P6 is excluded)
symmetric_machine = prepare_up_down_symmetric_machine(
    machine_data["active_coils"],
    machine_data["passive_coils"],
    limiter_data=machine_data["limiter"],
    wall_data=machine_data["wall"],
    z_midplane="auto",
    exclude_odd_active=True,
    verbose=True,
)

In [ ]:
# build machine
tokamak = build_machine.tokamak(
    active_coils_data=symmetric_machine.even_active_coils_data,
    passive_coils_data=symmetric_machine.passive_coils_data,
    limiter_data=symmetric_machine.limiter_data,
    wall_data=symmetric_machine.wall_data,
)

In [ ]:
# plot machine (notice the P6 coils now missing, around (R,Z)=(1.25, +/-1.1))
fig, ax = plt.subplots(figsize=(4, 8), dpi=80)
tokamak.plot(axis=ax, show=False)
ax.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)
ax.set(xlabel="Major radius, R [m]", ylabel="Height, Z [m]", aspect="equal")
ax.grid(zorder=0, alpha=0.75)
plt.show()

### 2. Construct a symmetric initial equilibrium

Symmetrising the machine changes it slightly, so the initial equilibrium of Example 11 must be solved again on the new machine. The upper and lower coils of every retained circuit are in series, so the circuit currents of Example 11 (and hence the PCS current limits defined for them) apply unchanged, and we simply repeat its forward solve with the same currents and plasma profiles, forcing the solution to be even in $Z$. (An inverse solve would also work, but would need new shape constraints and would give different coil currents.) The tiny current in `P6` is dropped with the circuit.

In [ ]:
# initialise eq (same grid as Example 11)
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=65,
)

# initialise profiles (same as Example 11)
profiles = Lao85(
    eq=eq,
    Ip=7.59e5,
    fvac=-0.52,
    alpha=[362685, 17696],
    beta=[0.103, 0.753],
    alpha_logic=True,
    beta_logic=True,
)

# initial coil currents (same as Example 11; the passive structures start with no current)
coil_currents = np.array(
    [
        2766.10782056, 193.12768191, 4019.86873871, 4857.42357466,
        -726.99282376, -1696.92521166, -77.7252396, 266.38207493,
        -73.01483716, -4169.86874271, -4518.28147417,
    ]
)
active_coils = list(tokamak.coils_list[: tokamak.n_active_coils])
for name, current in zip(active_coils, coil_currents):
    tokamak.set_coil_current(coil_label=name, current_value=current)

# static solver
static_solver = GSstaticsolver.NKGSsolver(eq)

In [ ]:
# the numerical operators need exactly up-down symmetric resistances and inductances, so these calls enforce this
eq._vgreen = symmetric_machine.symmetrise_even_machine_greens(eq._vgreen)
tokamak.coil_resist = symmetric_machine.symmetrise_even_machine_resistances(
    tokamak.coil_resist
)
tokamak.coil_self_ind = symmetric_machine.symmetrise_even_machine_square_operator(
    tokamak.coil_self_ind
)

In [ ]:
# solve for the equilibrium (even in Z)
static_solver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-8,
    force_up_down_symmetric=True,
    verbose=False,
)

We should have a nice up-down symmetric, diverted equilibrium in the symmetric machine, with the same X-point radius, inner/outer radii, and coil currents as Example 11.

In [ ]:
# plot the equilibrium
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(zorder=0, alpha=0.75)
ax1.set_aspect("equal")
eq.tokamak.plot(axis=ax1, show=False)                                                          # plots the active coils and passive structures
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color="k", linewidth=1.2, facecolor="w", zorder=0)    # plots the limiter
eq.plot(axis=ax1, show=False)                                                                  # plots the equilibrium
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()
plt.show()

If needed we can do some checks to ensure everything is fully up-down symmetric (i.e. machine operators commute with the metal-current reflection, the limiter mask is reflection-invariant, and the initial plasma fields and metal currents are even to round-off).

In [ ]:
passive_reflection_operator = symmetric_machine.passive_reflection_operator
reflection_operator = symmetric_machine.even_machine_reflection_operator


def relative_reflection_error(values, parity=1):
    denominator = np.linalg.norm(values)
    if denominator == 0:
        return 0.0
    return np.linalg.norm(values - parity * values[:, ::-1]) / denominator


def relative_commutator_error(matrix):
    numerator = np.linalg.norm(
        reflection_operator @ matrix - matrix @ reflection_operator
    )
    return numerator / np.linalg.norm(matrix)


reflected_greens = np.einsum(
    "ij,jrz->irz", reflection_operator, eq._vgreen[:, :, ::-1]
)
resistance_matrix = np.diag(tokamak.coil_resist)
initial_currents = tokamak.getCurrentsVec()
symmetry_errors = {
    "sampled Green functions": (
        np.linalg.norm(eq._vgreen - reflected_greens)
        / np.linalg.norm(eq._vgreen)
    ),
    "inductance operator": relative_commutator_error(tokamak.coil_self_ind),
    "resistance operator": relative_commutator_error(resistance_matrix),
    "plasma flux": relative_reflection_error(eq.plasma_psi),
    "toroidal current density": relative_reflection_error(profiles.jtor),
    "metal currents": (
        np.linalg.norm(reflection_operator @ initial_currents - initial_currents)
        / np.linalg.norm(initial_currents)
    ),
}
limiter_defects = np.count_nonzero(
    eq.limiter_handler.mask_inside_limiter
    != eq.limiter_handler.mask_inside_limiter[:, ::-1]
)

for name, error in symmetry_errors.items():
    print(f"{name:27s}: {error:.3e}")
print(f"{'limiter-mask defects':27s}: {limiter_defects}")
assert max(symmetry_errors.values()) < 1e-12
assert limiter_defects == 0

### 3. Build the virtual circuits for the PCS

We use FreeGSNKE's `VirtualCircuitHandling` class (see Example 09) to build the virtual circuits (VCs) for the ten shaping circuits (all circuits except the solenoid). It computes, by finite differences of static GS solves, the shape sensitivity matrix

$$
S_{kj} = \frac{\partial s_k}{\partial I_j},
$$

where $s_k$ are the shape targets (`Rin`, `Rout`, `Rx`, `Zx`, `Rsp`, defined below) and $I_j$ the shaping-circuit currents, and its pseudo-inverse $S^+$ (the VCs), which maps requested shape corrections to coil-current commands. The solenoid is excluded because it is reserved for plasma-current control.

In addition to the four shape targets of Example 11 (`Rin`, `Rout`, `Rx`, `Zx`), we include the radial position of the lower outer divertor strikepoint, `Rsp`. To extract it from an equilibrium object, `lower_outer_strikepoint(equilibrium)` calls `equilibrium.strikepoints()`, which returns the $(R, Z)$ intersections of the $\psi_{\mathrm{boundary}}$ contour with `tokamak.wall`. For a symmetric double-null diverted plasma this returns four points (inner and outer strikepoints in the lower and upper divertors); filtering to the lower divertor ($Z < 0$) and taking the point with largest $R$ selects the lower outer divertor tile intersection.

In [ ]:
def lower_outer_strikepoint(equilibrium):
    """Return the (R, Z) coordinates of the lower outer divertor strikepoint."""
    strikes = np.atleast_2d(equilibrium.strikepoints())
    lower_strikes = strikes[strikes[:, 1] < 0.0]
    return lower_strikes[np.argmax(lower_strikes[:, 0])]


# functions for the plasma descriptors (shape functions), extending Example 11 with Rsp
def plasma_descriptors(equilibrium):
    """Return the five shape measurements used by the controller: Rin, Rout, Rx, Zx, Rsp."""
    # inboard/outboard midplane radii
    rin_rout = equilibrium.innerOuterSeparatrix()

    # find the lower X-point: define a "box" in which to search for it
    xpt_box = [[0.33, -0.88], [0.95, -1.38]]
    xpt_mask = (
        (equilibrium.xpt[:, 0] >= xpt_box[0][0])
        & (equilibrium.xpt[:, 0] <= xpt_box[1][0])
        & (equilibrium.xpt[:, 1] <= xpt_box[0][1])
        & (equilibrium.xpt[:, 1] >= xpt_box[1][1])
    )
    xpts = equilibrium.xpt[xpt_mask, 0:2].reshape(-1, 2)
    # if there is more than one candidate, take the one closest to the O-point
    nearest = np.argmin(np.linalg.norm(xpts - equilibrium.opt[0, 0:2], axis=1))
    rx, zx = xpts[nearest]

    # lower outer divertor strikepoint radius
    rsp, _ = lower_outer_strikepoint(equilibrium)

    return np.array([rin_rout[0], rin_rout[1], rx, zx, rsp])


# names of the shape targets
ctrl_targets = ["Rin", "Rout", "Rx", "Zx", "Rsp"]

# calculate initial values
baseline_descriptors = plasma_descriptors(eq)

print("Active circuit order:", active_coils)
print("Shape parameters:", ctrl_targets)
print("Initial descriptors [m]:", baseline_descriptors)

In [ ]:
from freegsnke import virtual_circuits

# build the VCs (all shape targets are included in the calculation, the solenoid is excluded)
shape_coils = active_coils[1:]
VCs = virtual_circuits.VirtualCircuitHandling()
VCs.define_solver(static_solver)
VCs.calculate_VC(
    eq=eq,
    profiles=profiles,
    coils=shape_coils,
    target_names=ctrl_targets,
    target_calculator=plasma_descriptors,
    n_vc_workers=2,
    verbose=True,
    name="VC_shape",
)

shape_matrix = VCs.VC_shape.shape_matrix  # [no. of targets x no. of coils], units m/A
shape_vcs = VCs.VC_shape.VCs_matrix  # [no. of coils x no. of targets], units A/m

### 4. Configure the plasma-control loop

Here we set up the FPDT controllers as in Example 11 (with the same references, gains, limits, and time window), adding feedback control of the lower outer strikepoint radius `Rsp`, and omitting the vertical-control circuit. The controllers:

- hold the plasma current at its initial value ($7.59 \times 10^5\text{ A}$), counteracting its resistive decay;
- hold `Rin` and `Rout` at $0.28\text{ m}$ and $1.35\text{ m}$;
- ramp the lower X-point radius, $R_x$, from $0.57\text{ m}$ inwards to $0.50\text{ m}$ and back again;
- leave the lower X-point height, $Z_x$, uncontrolled (its blend is zero, so its virtual circuit is set to zero, as in Example 11);
- hold the lower outer strikepoint radius, $R_{\mathrm{sp}}$, at its initial value ($0.74\text{ m}$) for $50\text{ ms}$, sweep it linearly over $150\text{ ms}$ out to a Super-X position at $1.40\text{ m}$ ($t = 200\text{ ms}$), and hold it there for the remainder of the simulation.

The shape controllers use proportional-only control (`k_int` = 0), as in Example 11. Note that the equilibrium starts a few millimetres away from the `Rin`, `Rout`, and `Rx` references.

In [ ]:
from freegsnke.control_loop.pcs import PlasmaControlSystem

# simulation window (as in Example 11)
tmin = 0.0
tmax = 0.25
n_active = tokamak.n_active_coils
zeros = {"times": np.array([tmin, tmax]), "vals": np.zeros(2)}

# PLASMA CATEGORY (as in Example 11): hold Ip at its initial value with PID feedback only
plasma_data = {
    "ip_ref": {"times": np.array([tmin, tmax]), "vals": np.full(2, profiles.Ip)},
    "vloop_ff": zeros,
    "ip_blend": {"times": np.array([tmin, tmax]), "vals": np.ones(2)},
    "k_prop": {"times": np.array([tmin]), "vals": np.array([-5 * 4])},
    "k_int": {"times": np.array([tmin]), "vals": np.array([-50 * 2])},
    "k_deriv": {"times": np.array([tmin]), "vals": np.array([0])},
    "M_solenoid": {"times": np.array([tmin]), "vals": np.array([1.0])},
}


# SHAPE CATEGORY (as in Example 11, plus Rsp sweep to Super-X)
def shape_waveforms(ref_times, ref_values, k_prop, blend=1.0):
    """Return the waveforms of one shape target (a proportional-only controller)."""
    return {
        "ff": zeros,
        "ref": {"times": np.array(ref_times), "vals": np.array(ref_values)},
        "blend": {"times": np.array([tmin, tmax]), "vals": np.full(2, blend)},
        "k_prop": {"times": np.array([tmin]), "vals": np.array([k_prop])},
        "k_int": {"times": np.array([tmin]), "vals": np.array([0.0])},
        "k_deriv": {"times": np.array([tmin]), "vals": np.array([0.0])},
    }


rsp_initial = np.round(baseline_descriptors[ctrl_targets.index("Rsp")], 2)
shape_data = {
    "Rin": shape_waveforms([tmin, tmax], [0.28, 0.28], 120),
    "Rout": shape_waveforms([tmin, tmax], [1.35, 1.35], 170),
    "Rx": shape_waveforms(
        [tmin, 0.025, 0.075, 0.175, 0.225, tmax],
        [0.57, 0.57, 0.5, 0.5, 0.57, 0.57],
        120,
    ),
    "Zx": shape_waveforms([tmin, tmax], [-1.24, -1.24], 120, blend=0.0),  # uncontrolled
    "Rsp": shape_waveforms(
        [tmin, 0.05, 0.20, tmax],
        [rsp_initial, rsp_initial, 1.40, 1.40],
        120,
    ),
}

### Virtual circuits and electrical limits

The shape virtual circuits are the columns of $S^+$ (padded with zero on the solenoid), while the plasma-current virtual circuit acts solely on the solenoid. As in Example 11, the `Zx` virtual circuit is zero. The PCS current and voltage limits are copied from Example 11 (the same eleven circuits in the same order), including its limits that keep the `P4` and `P5` currents negative. Resistances and inductances are taken directly from the symmetrised tokamak object.

In [ ]:
# VIRTUAL CIRCUITS CATEGORY
circuits_data = {"coil_order": active_coils}
for target_index, target in enumerate(ctrl_targets):
    circuit = np.zeros(n_active)
    if target != "Zx":  # Zx is not controlled
        circuit[1:] = shape_vcs[:, target_index]
    circuits_data[target] = {"times": np.array([tmin]), "vals": [circuit]}
circuits_data["plasma"] = {
    "times": np.array([tmin]),
    "vals": [np.r_[1.0, np.zeros(n_active - 1)]],
}
for coil_name in active_coils:  # no feedforward coil-current drives
    circuits_data[f"{coil_name}_ref"] = zeros

# SYSTEMS CATEGORY (limits as in Example 11)
initial_active = tokamak.getCurrentsVec()[:n_active]
min_current_lims = np.array([-10000, -7000, -9000, -9000, -9000, -9000, -9000, -9000, -9000, -10000, -10000], dtype=float)
max_current_lims = np.array([10000, 7000, 9000, 9000, 9000, 9000, 9000, 9000, 9000, 0, 0], dtype=float)
systems_data = {
    "min_coil_curr_lims": {"times": [tmin], "vals": [min_current_lims]},
    "max_coil_curr_lims": {"times": [tmin], "vals": [max_current_lims]},
    "max_coil_curr_ramp_lims": {"times": [tmin], "vals": [1e12 * np.ones(n_active)]},
}
for coil_name in active_coils:
    systems_data[f"{coil_name}_pert"] = zeros

# PF CATEGORY (gains and voltage limits as in Example 11)
voltage_lims = np.array([2000.0] + [750.0] * (n_active - 1))
pf_data = {
    "R_matrix": {"times": [tmin], "vals": [tokamak.coil_resist[:n_active]]},
    "M_FF_matrix": {
        "times": [tmin],
        "vals": [tokamak.coil_self_ind[:n_active, :n_active]],
    },
    "M_FB_matrix": {
        "times": [tmin],
        "vals": [tokamak.coil_self_ind[:n_active, :n_active]],
    },
    "coil_gains": {"times": [tmin], "vals": [0.015 * np.ones(n_active)]},
    "coil_voltage_lims": {"times": np.array([tmin]), "vals": [voltage_lims]},
    "coil_voltage_slew_lims": {"times": [tmin], "vals": [1e12 * np.ones(n_active)]},
}

# VERTICAL CATEGORY: no vertical-control circuit (P6 is odd and was excluded), so no vertical feedback
vertical_data = {
    "z_ref": {"times": np.array([tmin, tmax]), "vals": np.zeros(2)},
    "k_prop": {"times": np.array([tmin]), "vals": np.array([0.0])},
    "k_deriv": {"times": np.array([tmin]), "vals": np.array([0.0])},
}

# COIL ACTIVATIONS CATEGORY: all coils active throughout
activation_data = {
    f"{name}_activation": {"times": np.array([tmin]), "vals": np.array([1.0])}
    for name in active_coils
}

### Assemble the PCS

The standard `PCS` interface is unchanged: symmetry reduction is handled entirely by the evolutive solver.

In [ ]:
# build
pcs = PlasmaControlSystem(
    plasma_data=plasma_data,
    shape_data=shape_data,
    shape_control_mode="PID",
    circuits_data=circuits_data,
    systems_data=systems_data,
    pf_data=pf_data,
    vertical_data=vertical_data,
    coil_activation_data=activation_data,
    active_coils=active_coils,
    ctrl_coils=active_coils,
    vertical_coils=[],
    ctrl_targets=ctrl_targets,
    plasma_target=["plasma"],
)

### 5. Initialise the reduced evolutive solver

In Example 10 we showed that the metals and the plasma are governed by the circuit equation (2) and the plasma equation (3). Linearising the plasma current density about the current equilibrium (equation (4)) gives the state-matrix system

$$
\mathbf{M}\dot{\mathbf{x}} + \mathbf{R}\mathbf{x} = \mathbf{c}, \qquad \mathbf{x} = \begin{bmatrix} \vec{I}_m \\ I_p \end{bmatrix}, \qquad A = -\mathbf{R}^{-1}\mathbf{M},
$$

where the eigenvalues of $A$ are the timescales $\tau_i$ of the system ($\tau_i > 0$ unstable, $\tau_i < 0$ stable). The `nl_solver` used in Examples 05 and 11 evolves all of these modes (or a subset if needed). This includes the fast, vertically unstable mode, whose short timescale forces a small timestep ($0.5\text{ ms}$ in Example 11).

`SymmetryReducedSolver` removes that mode by exploiting the up-down symmetry of the machine and the equilibrium. Let $S_m$ be the reflection operator for the metal currents (it swaps each upper/lower partner) and $S_y$ the reflection of the plasma grid about $Z = 0$. For an exactly symmetric machine and equilibrium, reflected partners have equal resistances and inductances, and the Jacobians in equation (4) are even fields. The linearised system then commutes with reflection, $SA = AS$ with $S = \mathrm{diag}(S_m, 1)$, so the even and odd parts of the state evolve independently: an even perturbation never drives an odd response. The vertical instability is a vertical displacement of the plasma, which is odd, so it lives in the odd part and is removed by keeping only the even part.

#### How the passive modes are calculated

The passive structures obey the circuit equation (2) with no plasma and no applied voltage:

$$
M_{p,p}\dot{\vec{I}}_p + R_{p,p}\vec{I}_p = 0.
$$

We form $A_p = -R_{p,p}^{-1}M_{p,p}$ and solve the eigenproblem

$$
A_p\vec{v}_i = \tau_i\vec{v}_i.
$$

Each eigenvector $\vec{v}_i$ is a pattern of passive currents that decays as $e^{t/\tau_i}$. All $\tau_i$ are negative because the passive structures only dissipate energy. The modes are stored with their decay rates $1/|\tau_i|$ and sorted from lowest rate (slowest decay) upwards.

For $n=138$ passive elements, we obtain $n/2 = 69$ up-down symmetric pairs. The passive reflection operator $S_p$ is a permutation matrix, built from the pairing found by `prepare_up_down_symmetric_machine`, that swaps each upper/lower partner. For a pair $(u, l)$:

- the $S_p = +1$ eigenvector is $(e_u + e_l)/\sqrt{2}$: both partners carry the same current, giving flux that is **even** in $Z$.
- the $S_p = -1$ eigenvector is $(e_u - e_l)/\sqrt{2}$: the partners carry opposite currents, giving a field that is **odd** in $Z$. This is the kind of current that responds to vertical plasma motion.

The 69 pairs therefore give a 69-dimensional even subspace and a 69-dimensional odd subspace. A passive element on the midplane has no partner and contributes to the even subspace only (there is none in this machine).

We solve the eigenproblem separately in each subspace, so every mode has a definite parity, even when eigenvalues are degenerate. This requires $S_pA_p = A_pS_p$, which holds when the resistances and inductances are reflection-symmetric. The solver checks this and raises an error if it fails.

#### What the reduced solver evolves

We keep only (a subset of) the even modes. The metal currents are written as $\vec{I}_m = P\vec{\alpha}$, where the columns of $P$ are the active-coil unit vectors and the retained even passive modes (in the following cell, we will retain 30 of the 69 even modes, chosen by their coupling to the plasma as in Example 11). The odd modes are removed before the plasma coupling is built. The reduced state is then

$$
\mathbf{x}_+ = \begin{bmatrix} \vec{\alpha} \\ I_p \end{bmatrix} \qquad (11 \text{ active circuits} + 30 \text{ passive modes} + I_p = 42 \text{ values}).
$$

Substituting $\vec{I}_m = P\vec{\alpha}$ into equation (5), multiplying by $P^{-1}R_{m,m}^{-1}$, and dividing equation (6) by the lumped plasma resistance $R_p$ gives the same equations as Example 10 in the reduced state:

$$
\mathbf{M}_+\dot{\mathbf{x}}_+ + \mathbf{x}_+ = \mathbf{f},
$$

$$
\mathbf{M}_+ =
\begin{bmatrix}
P^{-1}R_{m,m}^{-1}\left( M_{m,m}P + M_{m,y}\dfrac{\partial \vec{I}_y}{\partial \vec{\alpha}} \right) &
P^{-1}R_{m,m}^{-1}M_{m,y}\dfrac{\partial \vec{I}_y}{\partial I_p} \\[2ex]
\dfrac{\hat{\vec{I}}_y^{T}}{R_p}\left( M_{y,m}P + M_{y,y}\dfrac{\partial \vec{I}_y}{\partial \vec{\alpha}} \right) &
\dfrac{\hat{\vec{I}}_y^{T}}{R_p}M_{y,y}\dfrac{\partial \vec{I}_y}{\partial I_p}
\end{bmatrix},
$$

$$
\mathbf{f} =
\begin{bmatrix} P^{-1}R_{m,m}^{-1}\vec{V}_m \\ 0 \end{bmatrix}
-
\begin{bmatrix} P^{-1}R_{m,m}^{-1}M_{m,y} \\[1ex] \dfrac{\hat{\vec{I}}_y^{T}}{R_p}M_{y,y} \end{bmatrix}
\dfrac{\partial \vec{I}_y}{\partial \vec{\theta}}\dot{\vec{\theta}}.
$$

The Jacobians $\partial\vec{I}_y/\partial\vec{\alpha}$, $\partial\vec{I}_y/\partial I_p$ and $\partial\vec{I}_y/\partial\vec{\theta}$ come from Grad-Shafranov solves forced to be even in $Z$, and are projected with $\Pi_+ = \tfrac{1}{2}(I + S_y)$ to remove odd round-off. The system is advanced with implicit Euler,

$$
\mathbf{x}_+(t + \Delta t) = (\mathbf{M}_+ + \Delta t\,\mathbf{I})^{-1}\left(\Delta t\,\mathbf{f} + \mathbf{M}_+\mathbf{x}_+(t)\right).
$$

After each step the equilibrium is updated with a Grad-Shafranov solve, and the Jacobians are rebuilt if the plasma current density has changed by more than the relinearisation threshold.

**Note:** Without the odd modes, the fastest remaining timescales are resistive, which allows a much larger timestep to be taken (recall the small timesteps used in Examples 05 and 11). This is exact only for the linearisation about an even equilibrium in an even machine: the plasma cannot move vertically, so it cannot be used for vertical displacement events or asymmetric scenarios. The timestep cannot be arbitrarily large, however: it must still resolve the retained dynamics, keep the linearisation accurate (the Jacobians are rebuilt whenever the plasma has changed significantly), and be small compared with the timescales and gains used in the virtual PCS.

In [ ]:
from freegsnke.symmetry_reduced_solve import SymmetryReducedSolver

# plasma resistivity [Ohm m] (as in Example 11)
plasma_resistivity = 1e-7

# initialise the solver object
stepping = SymmetryReducedSolver(
    eq=eq,
    profiles=profiles,
    GSStaticSolver=static_solver,
    full_timestep=2.5e-3,      # <-- 2.5 ms timestep (5x that of Example 11)!
    max_internal_timestep=2.5e-3,
    plasma_resistivity=plasma_resistivity,
    mode_selection="coupling",
    fix_n_vessel_modes=30,     # <-- 30/69 retained even modes
    plasma_descriptor_function=plasma_descriptors,
    passive_reflection_operator=passive_reflection_operator,
    n_linearization_workers=4,
    verbose=False,
)

### Inspect passive-mode parity and truncation

Open markers show the discarded passive modes (even in blue, odd in orange); filled markers show the even modes retained after parity and mode-count selection (we keep the 30 even modes that couple most strongly to the plasma). The spatial panels show representative even and odd mode current patterns.

In [ ]:
normal_modes = stepping.evol_metal_curr.normal_modes
passive_parity = normal_modes.passive_mode_parity
passive_frequency = normal_modes.w_passive
retained_passive = stepping.evol_metal_curr.selected_modes_mask[n_active:]
passive_centres = np.array(
    [
        [np.mean(element["R"]), np.mean(element["Z"])]
        for element in symmetric_machine.passive_coils_data
    ]
)

representative_modes = {
    "even mode": np.flatnonzero(passive_parity > 0)[0],
    "odd mode": np.flatnonzero(passive_parity < 0)[0],
}
fig, axes = plt.subplots(1, 3, figsize=(13, 5))

for parity, label, colour in ((1, "even", "tab:blue"), (-1, "odd", "tab:orange")):
    parity_mask = (passive_parity == parity) & ~retained_passive
    axes[0].scatter(
        np.flatnonzero(parity_mask),
        passive_frequency[parity_mask],
        facecolors="none",
        edgecolors=colour,
        label=f"{label}, discarded",
    )
retained_indices = np.flatnonzero(retained_passive)
axes[0].scatter(
    retained_indices,
    passive_frequency[retained_passive],
    color="tab:blue",
    label="even, retained",
)
axes[0].set_yscale("log")
axes[0].set_xlabel("passive-mode index")
axes[0].set_ylabel("frequency [s$^{-1}$]")
axes[0].set_title("Parity-aware mode selection")
axes[0].grid(True)
axes[0].legend()

for axis, (title, mode_index) in zip(axes[1:], representative_modes.items()):
    amplitudes = normal_modes.Pmatrix[
        n_active:, n_active + mode_index
    ]
    scale = np.max(np.abs(amplitudes))
    points = axis.scatter(
        passive_centres[:, 0],
        passive_centres[:, 1],
        c=amplitudes,
        cmap="coolwarm",
        vmin=-scale,
        vmax=scale,
        s=30,
        edgecolor="k",
    )
    axis.axhline(0.0, color="0.6", linewidth=1)
    axis.set_aspect("equal")
    axis.set_xlabel("R [m]")
    axis.set_ylabel("Z [m]")
    axis.set_title(title)
    fig.colorbar(points, ax=axis, label="relative mode current", shrink=0.6)

fig.tight_layout()
plt.show()

### 6. Run the controlled evolution

Next, we'll run the simulation with a $2.5\text{ ms}$ solver step and a $1.25\text{ ms}$ PCS timestep (Example 11 used $0.5\text{ ms}$ and $0.25\text{ ms}$).

The virtual PCS will return the voltages to be applied to the active coils, which the reduced solver will use to advance the retained metal modes, plasma current, and profile response (and equilibrium!).

The PCS holds the plasma current at its initial value, ramps $R_x$ inwards and back, sweeps the lower outer strikepoint radius $R_{\mathrm{sp}}$ out to $1.40\text{ m}$ (Super-X), and holds `Rin` and `Rout` constant as described in Section 4. The plasma resistivity is set to $10^{-7}\,\Omega\,\mathrm{m}$, as in Example 11.

In [ ]:
# setup and storage
dt = stepping.dt_step
n_steps = int(round((tmax - tmin) / dt)) + 1  # simulate from tmin to tmax
dt_pcs = dt / 2
times = tmin + np.arange(n_steps) * dt

# keep four equilibrium snapshots, evenly spaced in time, for plotting later
snapshot_indices = np.linspace(0, n_steps - 1, 4).astype(int)
snapshots = {0: stepping.eq1.create_auxiliary_equilibrium()}

# re-linearisation: the solver re-linearises when the relative change in the toroidal current density
# since the last linearisation reaches this threshold (we store this criterion at every step)
relinearise_threshold = 0.05
relinearise_criterion = np.zeros(n_steps - 1)

dynamic_currents = np.zeros((len(stepping.vessel_currents_vec), n_steps))
dynamic_ip = np.zeros(n_steps)
dynamic_shape = np.zeros((len(ctrl_targets), n_steps))
dynamic_z = np.zeros(n_steps)
dynamic_limited = np.zeros(n_steps, dtype=bool)  # is the plasma limited (not diverted)?
voltages = np.zeros((n_active, n_steps - 1))
step_times = np.zeros(n_steps - 1)

dynamic_currents[:, 0] = stepping.vessel_currents_vec
dynamic_ip[0] = stepping.profiles1.Ip
dynamic_shape[:, 0] = plasma_descriptors(stepping.eq1)
dynamic_z[0] = stepping.eq1.Zcurrent()
dynamic_limited[0] = bool(stepping.profiles1.flag_limiter)

ip_hist_prev = 0.0
ip_err_prev = 0.0
shape_error_prev = np.zeros(len(ctrl_targets))
shape_history_prev = np.zeros(len(ctrl_targets))
approved_current_prev = initial_active.copy()
approved_voltage_prev = np.zeros(n_active)

In [ ]:
# simulate!
for index, time_value in enumerate(times[:-1]):

    print("-----")
    print(f"t = {np.round(time_value,5)}s (step {index+1}/{n_steps-1})")

    result = pcs.calculate_ctrl_voltages(
        t=time_value,
        dt=dt_pcs,
        dt_simulator=dt,
        ip_meas=dynamic_ip[index],
        ip_hist_prev=ip_hist_prev,
        ip_err_prev=ip_err_prev,
        T_meas=dynamic_shape[:, index],
        T_err_prev=shape_error_prev,
        T_hist_prev=shape_history_prev,
        I_approved_prev=approved_current_prev,
        I_meas=dynamic_currents[:n_active, index],
        V_approved_prev=approved_voltage_prev,
        zip_meas=dynamic_z[index] * dynamic_ip[index],
        zipv_meas=0.0,
        active_coil_resists=tokamak.coil_resist[:n_active],
        verbose=False,
    )
    (
        approved_voltage,
        ip_hist_prev,
        ip_err_prev,
        shape_error_prev,
        shape_history_prev,
        approved_current_prev,
        active_resistances,
    ) = result
    voltages[:, index] = approved_voltage
    approved_voltage_prev = approved_voltage.copy()

    start = time.perf_counter()
    stepping.nlstepper(
        plasma_resistivity=plasma_resistivity,
        active_voltage_vec=approved_voltage,
        # Lao logic adds one dependent closure coefficient to each stored
        # array; the evolutive interface accepts the independent values only.
        profiles_parameters={
            "alpha": profiles.alpha[:-1].copy(),
            "beta": profiles.beta[:-1].copy(),
        },
        custom_active_coil_resistances=active_resistances,
        linear_only=True,
        target_relative_tol_currents=1e-2,
        target_relative_tol_GS=1e-2,
        working_relative_tol_GS=5e-2,
        max_solving_iterations=20,
        relinearise_threshold=relinearise_threshold,
        no_GS=False,
        verbose=False,
    )
    step_times[index] = time.perf_counter() - start
    relinearise_criterion[index] = stepping.relinearise_criteria

    dynamic_currents[:, index + 1] = stepping.vessel_currents_vec
    dynamic_ip[index + 1] = stepping.profiles1.Ip
    dynamic_shape[:, index + 1] = plasma_descriptors(stepping.eq1)
    dynamic_z[index + 1] = stepping.eq1.Zcurrent()
    dynamic_limited[index + 1] = bool(stepping.profiles1.flag_limiter)
    if index + 1 in snapshot_indices:
        snapshots[index + 1] = stepping.eq1.create_auxiliary_equilibrium()

    # print some stuff to track solve
    print(f"      PLASMA QUANTITIES")
    print(f"        Ip = {np.round(dynamic_ip[index+1]/1000,1)} [kA]")
    print(f"        Shape parameters {ctrl_targets} = {np.round(dynamic_shape[:,index+1],3)}")
    print(f"        Z position = {np.round(dynamic_z[index+1],5)} [m]")

### 7. Inspect the results

Below we plot the plasma current, the plasma vertical position $Z_{\mathrm{current}}$ (held on the midplane by symmetry), and the five shape quantities (`Rin`, `Rout`, `Rx`, `Zx`, and the lower outer strikepoint radius `Rsp`) against their reference targets, followed by the active coil currents and voltages (with the PCS limits), snapshots of the equilibrium as it evolves into the Super-X configuration, and finally the passive structure currents.

In [ ]:
t_ms = 1e3 * times
shape_reference = np.array(  # reference waveform of each shape target, evaluated at the simulation times
    [
        np.interp(
            times,
            pcs.ShapeController.data[target]["ref"]["times"],
            pcs.ShapeController.data[target]["ref"]["vals"],
        )
        for target in ctrl_targets
    ]
)
t_voltage_ms = 1e3 * times[:-1]  # each voltage request is applied from the start of its step
colours = {name: f"C{index}" for index, name in enumerate(active_coils)}

# plasma current, vertical position, and shape: evolved values against their reference targets
ip_reference = np.interp(times, plasma_data["ip_ref"]["times"], plasma_data["ip_ref"]["vals"])
z_reference = np.interp(times, vertical_data["z_ref"]["times"], vertical_data["z_ref"]["vals"])
fig, axes = plt.subplots(2, 4, figsize=(15, 6), dpi=80, sharex=True, constrained_layout=True)
ip_axis = axes[0, 0]
z_axis = axes[0, 1]
shape_axes = [axes[0, 2], axes[0, 3], axes[1, 0], axes[1, 1], axes[1, 2]]
axes[1, 3].axis("off")  # unused

ip_axis.plot(t_ms, dynamic_ip / 1e3, color="k", label="evolved")
ip_axis.plot(t_ms, ip_reference / 1e3, color="tab:red", linestyle="--", label="reference")
ip_axis.set_ylabel(r"Plasma current, $I_p$ [kA]")
ip_axis.legend()

z_axis.plot(t_ms, dynamic_z, color="k", label="evolved")
z_axis.plot(t_ms, z_reference, color="tab:red", linestyle="--", label="reference")
z_axis.set_ylabel(r"Vertical position, $Z$ [m]")

for target_index, (axis, target) in enumerate(zip(shape_axes, ctrl_targets)):
    axis.plot(t_ms, dynamic_shape[target_index], color="k", label="evolved")
    if shape_data[target]["blend"]["vals"][0] > 0:  # the reference of an uncontrolled target is not followed
        axis.plot(t_ms, shape_reference[target_index], color="tab:red", linestyle="--", label="reference")
        axis.set_ylabel(f"{target} [m]")
    else:
        axis.set_ylabel(f"{target} [m] (uncontrolled)")

for axis in [ip_axis, z_axis] + shape_axes:
    axis.grid(alpha=0.5)
for axis in (axes[0, 3], axes[1, 0], axes[1, 1], axes[1, 2]):
    axis.tick_params(labelbottom=True)
    axis.set_xlabel("Time [ms]")
fig.suptitle("Plasma current, vertical position, and shape: evolved against reference")
plt.show()

# active coil currents and voltages (solenoid and shape coils are shown separately)
solenoid = active_coils.index("Solenoid")
shape_coil_indices = [index for index in range(n_active) if index != solenoid]
voltage_limits = pf_data["coil_voltage_lims"]["vals"][0]

fig, axes = plt.subplots(2, 3, figsize=(15, 6), dpi=80, sharex=True, constrained_layout=True)

axes[0, 0].plot(t_ms, dynamic_currents[solenoid] / 1e3, color=colours["Solenoid"], label="Solenoid")
axes[0, 0].axhline(max_current_lims[solenoid] / 1e3, color="tab:red", linestyle="--", linewidth=1, label="PCS limits")
axes[0, 0].axhline(min_current_lims[solenoid] / 1e3, color="tab:red", linestyle="--", linewidth=1)
axes[0, 0].set_ylabel("Solenoid current [kA]")
axes[0, 0].legend()

axes[1, 0].step(t_voltage_ms, voltages[solenoid], where="post", color=colours["Solenoid"])
axes[1, 0].axhline(voltage_limits[solenoid], color="tab:red", linestyle="--", linewidth=1, label="PCS limits")
axes[1, 0].axhline(-voltage_limits[solenoid], color="tab:red", linestyle="--", linewidth=1)
axes[1, 0].set_ylabel("Solenoid voltage [V]")
axes[1, 0].legend()

for index in shape_coil_indices:
    name = active_coils[index]
    axes[0, 1].plot(t_ms, dynamic_currents[index] / 1e3, color=colours[name], label=name)
    axes[0, 2].plot(t_ms, (dynamic_currents[index] - dynamic_currents[index, 0]) / 1e3, color=colours[name], label=name)
    axes[1, 1].step(t_voltage_ms, voltages[index], where="post", color=colours[name], label=name)
    axes[1, 2].step(t_voltage_ms, voltages[index], where="post", color=colours[name], label=name)
axes[1, 1].axhline(voltage_limits[shape_coil_indices[0]], color="tab:red", linestyle="--", linewidth=1)
axes[1, 1].axhline(-voltage_limits[shape_coil_indices[0]], color="tab:red", linestyle="--", linewidth=1)
axes[0, 1].set_ylabel("Shape-coil currents [kA]")
axes[0, 2].set_ylabel("Shape-coil current change [kA]")
axes[1, 1].set_ylabel("Shape-coil voltages [V] (limits: dashed)")
axes[1, 2].set_ylabel("Shape-coil voltages [V] (zoom)")
axes[1, 2].set_ylim(-250, 250)
axes[0, 1].legend(ncol=2, fontsize="small")

for axis in axes.ravel():
    axis.grid(alpha=0.5)
for axis in axes[1]:
    axis.set_xlabel("Time [ms]")
fig.suptitle("Active coil currents and voltages")
plt.show()

The next cell quantifies how closely the controllers follow their references (the shape errors are measured against the reference waveforms, and the initial offsets of a few millimetres in `Rin`, `Rout`, and `Rx` are included), and whether the coil currents and voltages stay within the PCS limits.

In [ ]:
# tracking errors against the reference waveforms
ip_error = 100 * np.abs(dynamic_ip - ip_reference) / ip_reference
print(f"Ip: maximum deviation {np.max(ip_error):.2f}% (final {ip_error[-1]:.2f}%)")
for target_index, target in enumerate(ctrl_targets):
    if shape_data[target]["blend"]["vals"][0] > 0:
        error = np.abs(dynamic_shape[target_index] - shape_reference[target_index])
        print(f"{target}: maximum error {1e3 * np.max(error):.1f} mm, rms {1e3 * np.sqrt(np.mean(error**2)):.1f} mm")
    else:
        drift = 1e3 * np.max(np.abs(dynamic_shape[target_index] - dynamic_shape[target_index, 0]))
        print(f"{target} (uncontrolled): drift from the initial value {drift:.1f} mm")
print(f"Vertical position of the plasma: maximum |Z| = {np.max(np.abs(dynamic_z)):.1e} m")
print(f"Plasma limited at any time: {bool(np.any(dynamic_limited))}")

# limits (the metal currents of the active coils are the first n_active entries)
active_currents = dynamic_currents[:n_active]
current_ok = np.all((active_currents >= min_current_lims[:, None]) & (active_currents <= max_current_lims[:, None]))
margin = np.minimum(max_current_lims[:, None] - active_currents, active_currents - min_current_lims[:, None]).min(axis=1)
print(f"Coil currents within PCS limits: {current_ok} (smallest margin: {active_coils[np.argmin(margin)]}, {margin.min():.0f} A)")
print(f"Coil voltages within PCS limits: {bool(np.all(np.abs(voltages) <= voltage_limits[:, None] + 1e-9))}")
saturated = np.any(np.isclose(np.abs(voltages), voltage_limits[:, None]), axis=1)
print(f"Coils whose voltage saturates at some time: {[name for name, flag in zip(active_coils, saturated) if flag]}")

In [ ]:
from matplotlib.lines import Line2D

# pre-compute outer strikepoints before creating the figure, because
# equilibrium.strikepoints() internally calls plt.contour(...) and plt.close(),
# which would close the active figure if called inside the plotting loop
snapshot_strikepoints = {
    index: lower_outer_strikepoint(snapshots[index])
    for index in snapshot_indices
}

# evolution of the equilibrium: four snapshots, evenly spaced in time
fig, axes = plt.subplots(1, 4, figsize=(13, 7), dpi=80, sharex=True, sharey=True, constrained_layout=True)
for axis, index in zip(axes, snapshot_indices):
    snapshot = snapshots[index]
    snapshot.tokamak.plot(axis=axis, show=False)  # plots the active coils and passive structures
    snapshot.plot(axis=axis, show=False)  # plots the equilibrium and its separatrix
    rsp_snap, zsp_snap = snapshot_strikepoints[index]
    axis.plot([rsp_snap, rsp_snap], [zsp_snap, -zsp_snap], "ko", markersize=5, zorder=5)
    axis.set_title(f"t = {1e3 * times[index]:.0f} ms")
    axis.grid(alpha=0.5)
for axis in axes[1:]:
    axis.set_ylabel("")
axes[0].set_xlim(0.1, 2.15)
axes[0].set_ylim(-2.25, 2.25)
axes[-1].legend(
    handles=[
        Line2D([], [], color="r", label="separatrix"),
        Line2D([], [], color="r", marker="x", linestyle="", label="X-points"),
        Line2D([], [], color="g", marker="2", linestyle="", label="O-point"),
        Line2D([], [], color="k", marker="o", linestyle="", label="outer strikepoints"),
    ],
    loc="lower right",
    fontsize="small",
)

plt.show()

In [ ]:
# passive structure currents (the passive elements follow the active coils in the stored current vector)
passive_currents = dynamic_currents[n_active:]
passive_names = list(symmetric_machine.passive_names)
upper = np.array([passive_names.index(pair.upper_name) for pair in symmetric_machine.passive_pairs])
lower = np.array([passive_names.index(pair.lower_name) for pair in symmetric_machine.passive_pairs])
passive_centres = np.array([[np.mean(element["R"]), np.mean(element["Z"])] for element in symmetric_machine.passive_coils_data])
scale, unit = (1e-3, "kA") if np.max(np.abs(passive_currents)) > 1e3 else (1.0, "A")

fig, (time_axis, space_axis) = plt.subplots(1, 2, figsize=(13, 6), dpi=80, constrained_layout=True)

# time evolution: reflected partners carry identical currents, so only one element of each pair is drawn
largest = np.argsort(np.max(np.abs(passive_currents[upper]), axis=1))[::-1]
for rank, pair_index in enumerate(largest):
    label = passive_names[upper[pair_index]] if rank < 8 else None  # label the eight largest only
    time_axis.plot(t_ms, scale * passive_currents[upper[pair_index]], linewidth=1, label=label)
time_axis.set_xlabel("Time [ms]")
time_axis.set_ylabel(f"Passive structure current [{unit}]")
time_axis.set_title("Passive currents (one element per pair; the partners are identical)")
time_axis.legend(ncol=2, fontsize="small")
time_axis.grid(alpha=0.5)

# spatial distribution at the end of the simulation
limit = scale * np.max(np.abs(passive_currents[:, -1]))
points = space_axis.scatter(
    passive_centres[:, 0],
    passive_centres[:, 1],
    c=scale * passive_currents[:, -1],
    cmap="coolwarm",
    vmin=-limit,
    vmax=limit,
    s=40,
    edgecolor="k",
    zorder=3,
)
space_axis.plot(tokamak.limiter.R, tokamak.limiter.Z, "k-", linewidth=1.2)
space_axis.set_aspect("equal")
space_axis.set_xlabel("Major radius [m]")
space_axis.set_ylabel("Height [m]")
space_axis.set_title(f"Passive currents at t = {t_ms[-1]:.0f} ms")
space_axis.grid(alpha=0.5)
fig.colorbar(points, ax=space_axis, label=f"Passive structure current [{unit}]", shrink=0.6)
plt.show()

upper_lower_difference = np.max(np.abs(passive_currents[upper] - passive_currents[lower]))
print(f"Maximum upper/lower passive current difference: {upper_lower_difference:.2e} A")

The passive structure currents are even in $Z$: each upper element carries exactly the same current as its lower partner. The largest changes occur when the $R_x$ and $R_{\mathrm{sp}}$ ramps start and end, and the largest currents flow in the vessel, divertor coil cases, and structures closest to the plasma and divertor legs.

Finally, we look at the wall-clock time taken by each step of the evolutive solver (this excludes the virtual PCS calculation). The red markers show the steps in which the solver re-linearised, i.e. where the relative change in the toroidal current density since the last linearisation reached the threshold (5%). Re-linearisations are expensive because the Jacobians are rebuilt, and steps where the equilibrium changes rapidly during the $R_x$ and $R_{\mathrm{sp}}$ ramps also require more Newton-Krylov iterations in the inner Grad-Shafranov solve.

In [ ]:
# wall-clock time of each evolutive solver step (the virtual PCS calculation is not included)
median_step = np.median(step_times)
t_step_ms = 1e3 * times[1:]  # time at the end of each step
relin_steps = np.flatnonzero(relinearise_criterion >= relinearise_threshold)  # steps in which the solver re-linearised

fig, (step_axis, total_axis) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=80, constrained_layout=True)

step_axis.semilogy(t_step_ms, step_times, color="k", linewidth=0.8, label="all steps")
step_axis.semilogy(t_step_ms[relin_steps], step_times[relin_steps], "v", color="tab:red", markersize=7, label=f"re-linearisations ({len(relin_steps)})")
step_axis.axhline(median_step, color="tab:blue", linestyle="--", linewidth=1, label=f"median step ({1e3 * median_step:.0f} ms)")
step_axis.set_xlabel("Time [ms]")
step_axis.set_ylabel("Wall-clock time per solver step [s]")
step_axis.set_title("Time per step")
step_axis.legend(loc="lower right")
step_axis.grid(alpha=0.5, which="both")

total_axis.plot(t_step_ms, np.cumsum(step_times), color="k")
for step in relin_steps:
    total_axis.axvline(t_step_ms[step], color="tab:red", linestyle=":", linewidth=1)
total_axis.plot([], [], color="tab:red", linestyle=":", label="re-linearisation")
total_axis.set_xlabel("Simulated time [ms]")
total_axis.set_ylabel("Cumulative wall-clock time [s]")
total_axis.set_title(f"Total: {np.sum(step_times):.0f} s for {t_step_ms[-1]:.0f} ms of plasma evolution")
total_axis.legend()
total_axis.grid(alpha=0.5)
plt.show()

print(f"{len(step_times)} steps of {1e3 * dt:.1f} ms | median {1e3 * median_step:.0f} ms | total {np.sum(step_times):.0f} s")
print(f"{len(relin_steps)} re-linearisations, at t = {np.round(1e3 * times[relin_steps], 1)} ms")